#Set up

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Função do gráfico

In [ ]:
def plot_setor_donut(
    df,
    col_categoria="Operadora",
    col_valor="Num_Usuarios_Unicos",
    titulo="Distribuição por categoria",
    moeda=False,
    largura_px=1000,
    altura_px=700,
    hole=0.55,
    mapa_cores=None,
    cor_padrao="rgb(200, 200, 200)",
    texto_fora_quando_moeda=True,
    texto_fora_se_muitas_categorias=6,
    exibir_total_centro=True,
    titulo_total_centro="Total",
    mostrar_tabela=False,
):
    """
    Gráfico de setor (donut) sem legenda, com rótulos completos nas fatias.
    """

    # =========================
    # 1. Validações Iniciais
    # =========================
    # Verifica se as colunas de categoria e valor existem no DataFrame.
    if col_categoria not in df.columns:
        raise ValueError(f"Coluna '{col_categoria}' não encontrada.")

    if col_valor not in df.columns:
        raise ValueError(f"Coluna '{col_valor}' não encontrada.")

    # Verifica se o DataFrame não está vazio.
    if df.empty:
        raise ValueError("DataFrame vazio.")

    # =========================
    # 2. Preparação dos Dados
    # =========================
    d = df.copy() # Cria uma cópia do DataFrame para evitar modificações no original.
    # Converte a coluna de valor para numérico, tratando erros com 'coerce' (valores inválidos viram NaN).
    d[col_valor] = pd.to_numeric(d[col_valor], errors="coerce")
    # Remove linhas onde a categoria ou o valor numérico são nulos.
    d = d.dropna(subset=[col_categoria, col_valor])

    # =========================
    # 3. Agregação e Cálculo de Percentual
    # =========================
    # Agrupa os dados pela coluna de categoria e soma os valores.
    d = (
        d.groupby(col_categoria, as_index=False)[col_valor]
        .sum()
    )

    # Calcula o total geral dos valores.
    total = d[col_valor].sum()

    # Verifica se o total é zero para evitar divisão por zero.
    if total == 0:
        raise ValueError("Total igual a zero.")

    # Calcula o percentual de cada categoria em relação ao total.
    d["percentual"] = d[col_valor] / total

    # =========================
    # 4. Definição de Cores
    # =========================
    # Inicializa o mapa de cores se não for fornecido.
    if mapa_cores is None:
        mapa_cores = {}

    # Atribui cores a cada categoria, usando 'mapa_cores' ou 'cor_padrao'.
    cores = [
        mapa_cores.get(cat, cor_padrao)
        for cat in d[col_categoria]
    ]

    # =========================
    # 5. Formatação de Valores e Textos
    # =========================
    # Função auxiliar para formatar números no padrão brasileiro (milhares com '.', decimais com ',').
    def fmt_ptbr(v, casas=2):
        s = f"{v:,.{casas}f}"
        return s.replace(",", "X").replace(".", ",").replace("X", ".")

    # Função auxiliar para formatar o valor principal, opcionalmente como moeda.
    def fmt_valor(v):
        if moeda:
            return f"R$ {fmt_ptbr(v, 2)}"
        return f"{int(round(v, 0)):,}".replace(",", ".") # Formata inteiro com separador de milhares.

    # Cria o texto completo para cada fatia do gráfico, incluindo categoria, valor formatado e percentual.
    d["texto"] = [
        f"<b>{cat}</b><br>{fmt_valor(v)}<br>({p:.1%})"
        for cat, v, p in zip(d[col_categoria], d[col_valor], d["percentual"])
    ]

    # =========================
    # 6. Regra para Posição do Texto
    # =========================
    # Verifica se há muitas categorias para decidir a posição do texto (dentro/fora).
    muitas_categorias = d[col_categoria].nunique() >= texto_fora_se_muitas_categorias

    # Define a posição do texto e o tamanho da fonte com base nas condições.
    if (texto_fora_quando_moeda and moeda) or muitas_categorias:
        textposition = "outside"
        textfont_size = 16
    else:
        textposition = "inside"
        textfont_size = 18

    # =========================
    # 7. Criação do Gráfico de Rosca (Donut)
    # =========================
    fig = go.Figure(
        data=[
            go.Pie(
                labels=d[col_categoria], # Rótulos para identificar as fatias.
                values=d[col_valor], # Valores que determinam o tamanho das fatias.
                text=d["texto"], # Texto personalizado exibido em cada fatia.
                textinfo="text", # Garante que o 'text' personalizado seja usado.
                textposition=textposition, # Posição do texto (dentro ou fora da fatia).
                textfont=dict(size=textfont_size), # Tamanho da fonte do texto.
                marker=dict(colors=cores), # Cores das fatias.
                hole=hole, # Define o tamanho do "buraco" central para criar o efeito de donut.
                sort=False, # Impede a ordenação automática das fatias.
            )
        ]
    )

    # =========================
    # 8. Exibição do Total no Centro
    # =========================
    if exibir_total_centro:
        fig.add_annotation(
            text=(
                f"<b>{titulo_total_centro}</b><br>" # Título do total no centro.
                f"<span style='font-size:20px'>{fmt_valor(total)}</span>" # Valor total formatado.
            ),
            x=0.5, # Posição X central.
            y=0.5, # Posição Y central.
            showarrow=False, # Não mostra seta para a anotação.
            align="center" # Alinhamento do texto no centro.
        )

    # =========================
    # 9. Configuração de Layout do Gráfico
    # =========================
    fig.update_layout(
        title=dict(
            text=f"<b>{titulo}</b>", # Título principal do gráfico.
            x=0.5, # Centraliza o título horizontalmente.
            xanchor="center", # Âncora do título no centro.
            font=dict(size=22) # Tamanho da fonte do título.
        ),
        paper_bgcolor="white", # Cor de fundo do papel.
        plot_bgcolor="white", # Cor de fundo da área de plotagem.
        width=largura_px, # Largura do gráfico em pixels.
        height=altura_px, # Altura do gráfico em pixels.
        showlegend=False,  # <--- Remove a legenda para um gráfico donut, já que o texto nas fatias é informativo.
        margin=dict(l=40, r=40, t=90, b=40) # Margens do gráfico (esquerda, direita, topo, base).
    )

    # =========================
    # 10. Exibição do Gráfico
    # =========================
    fig.show()

    # =========================
    # 11. Tabela Opcional
    # =========================
    if mostrar_tabela:
        # Exibe uma tabela com os dados plotados, ordenados por valor e com percentual formatado.
        display(
            d.sort_values(col_valor, ascending=False)
             .assign(percentual=lambda x: (x["percentual"] * 100).round(1)) # Converte percentual para 0-100.
        )

    # Retorna o objeto figura do Plotly e o DataFrame processado.
    return fig, d

# Dados de demostração

In [ ]:
df_teste_custo = pd.DataFrame({
    "Operadora": [
        "Bradesco", "Bradesco", "CNU", "Unimed"
    ],
    "Custo": [
        152340.75, 80320.20, 112540.00, 98500.40
    ]
})

In [ ]:
mapa_cores_operadoras = {
    "Bradesco": "rgb(235, 99, 115)",
    "CNU": "rgb(102, 187, 130)",
    "Unimed": "rgb(120, 170, 255)",
    "SulAmérica": "rgb(255, 193, 7)",
    "Amil": "rgb(160, 160, 160)"
}

# Plotagem

In [ ]:
fig, df_plotado = plot_setor_donut(
    df=df_teste_custo,
    col_categoria="Operadora",
    col_valor="Custo",
    titulo="Distribuição de custo por operadora",
    moeda=True,
    mapa_cores=mapa_cores_operadoras,
    mostrar_tabela=False
)